# Visualisation par point de données — `blob_nn_4x10-1`

Ce notebook affiche **chaque point de données individuellement** (pas de moyenne par
combo) : un point = une ligne du CSV = un `(combo, data_index)`.

**Colonnes attendues** :
`combo, strategy, l, u_idx, k, j_idx, data_index, target, certified, optimal_value, gain, LB_neuron1, UB_neuron1, LB_neuron2, UB_neuron2`


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# === CONFIGURATION ===
INPUT_CSV = "combo_detail_by_datapoint.csv"

COLUMNS = [
    "combo", "strategy", "l", "u_idx", "k", "j_idx",
    "data_index", "target", "certified", "optimal_value", "gain",
    "LB_neuron1", "UB_neuron1", "LB_neuron2", "UB_neuron2",
]

NUMERIC_COLS = [
    "l", "u_idx", "k", "j_idx", "data_index", "target",
    "optimal_value", "gain", "LB_neuron1", "UB_neuron1", "LB_neuron2", "UB_neuron2",
]

# Détection automatique de la présence d'un en-tête.
with open(INPUT_CSV) as f:
    first_line = f.readline().strip().split(",")

has_header = first_line[:2] == ["combo", "strategy"]

if has_header:
    df = pd.read_csv(INPUT_CSV)
    missing = set(COLUMNS) - set(df.columns)
    if missing:
        raise ValueError(f"Colonnes attendues manquantes dans {INPUT_CSV}: {missing}")
else:
    df = pd.read_csv(INPUT_CSV, header=None, names=COLUMNS)

for col in NUMERIC_COLS:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["certified"] = df["certified"].astype(str).str.lower().isin(["true", "1"])

print(f"En-tête détecté : {has_header}")
print(f"{len(df)} lignes (points de données), {df['combo'].nunique()} combo(s)")
df.head()


## 1. Gain par point de données, dans l'ordre du fichier\n\nChaque point = une ligne. Couleur = certifié ou non.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

colors = np.where(df["certified"], "#2a9d8f", "#e76f51")
ax.scatter(range(len(df)), df["gain"], c=colors, alpha=0.5, s=10)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("Index de ligne (ordre du fichier)")
ax.set_ylabel("Gain")
ax.set_title("Gain par point de données")

handles = [
    plt.Line2D([0], [0], marker="o", color="w", markerfacecolor="#2a9d8f", label="certifié", markersize=6),
    plt.Line2D([0], [0], marker="o", color="w", markerfacecolor="#e76f51", label="non certifié", markersize=6),
]
ax.legend(handles=handles)
plt.tight_layout()
plt.show()


## 2. Gain par point de données, par combo (nuage de points, un combo par ligne)\n\nChaque combo occupe une bande horizontale ; chaque point y est dispersé verticalement (jitter) juste pour éviter la superposition — la position verticale dans la bande n'a pas de sens en elle-même, seule la couleur/l'abscisse (gain) compte.

In [ ]:
# Pour la lisibilité, on peut limiter le nombre de combos affichés.
N_MAX_COMBOS = 40
combos = df["combo"].dropna().unique()
if len(combos) > N_MAX_COMBOS:
    print(f"[!] {len(combos)} combos détectés, affichage limité aux {N_MAX_COMBOS} premiers (triés par nom). "
          f"Changez N_MAX_COMBOS pour en voir plus.")
    combos = sorted(combos)[:N_MAX_COMBOS]
else:
    combos = sorted(combos)

subset = df[df["combo"].isin(combos)].copy()
combo_to_y = {c: i for i, c in enumerate(combos)}
subset["y"] = subset["combo"].map(combo_to_y) + np.random.uniform(-0.3, 0.3, size=len(subset))

fig, ax = plt.subplots(figsize=(12, max(4, 0.25 * len(combos))))
colors = np.where(subset["certified"], "#2a9d8f", "#e76f51")
ax.scatter(subset["gain"], subset["y"], c=colors, alpha=0.5, s=10)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_yticks(range(len(combos)))
ax.set_yticklabels(combos)
ax.set_xlabel("Gain")
ax.set_title("Gain — chaque point individuel, par combo")
ax.invert_yaxis()
plt.tight_layout()
plt.show()


## 3. Gain vs largeur des bornes des neurones (LB/UB)\n\nChaque point = un `(combo, data_index)`.

In [ ]:
plot_df = df.dropna(subset=["gain", "LB_neuron1", "UB_neuron1", "LB_neuron2", "UB_neuron2"]).copy()
plot_df["width_neuron1"] = plot_df["UB_neuron1"] - plot_df["LB_neuron1"]
plot_df["width_neuron2"] = plot_df["UB_neuron2"] - plot_df["LB_neuron2"]
plot_df["width_product"] = plot_df["width_neuron1"] * plot_df["width_neuron2"]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(plot_df["width_neuron1"], plot_df["gain"], alpha=0.35, s=10, label="neurone 1")
axes[0].scatter(plot_df["width_neuron2"], plot_df["gain"], alpha=0.35, s=10, label="neurone 2")
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set_xlabel("Largeur de l'intervalle [LB, UB]")
axes[0].set_ylabel("Gain")
axes[0].set_title("Gain vs largeur d'intervalle (par neurone)")
axes[0].legend()

axes[1].scatter(plot_df["width_product"], plot_df["gain"], alpha=0.35, s=10, color="#e76f51")
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_xlabel("Largeur neurone1 × Largeur neurone2")
axes[1].set_ylabel("Gain")
axes[1].set_title("Gain vs produit des largeurs")

plt.tight_layout()
plt.show()

corr1 = plot_df["width_neuron1"].corr(plot_df["gain"])
corr2 = plot_df["width_neuron2"].corr(plot_df["gain"])
corr_prod = plot_df["width_product"].corr(plot_df["gain"])
print(f"Corrélation gain / largeur neurone1 : {corr1:.3f}")
print(f"Corrélation gain / largeur neurone2 : {corr2:.3f}")
print(f"Corrélation gain / largeur produit  : {corr_prod:.3f}")


## 4. Gain vs data_index (position dans le test set), tous combos superposés

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
colors = np.where(df["certified"], "#2a9d8f", "#e76f51")
ax.scatter(df["data_index"], df["gain"], c=colors, alpha=0.4, s=10)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("data_index (position du point testé)")
ax.set_ylabel("Gain")
ax.set_title("Gain vs data_index — tous les combos, chaque point individuel")
plt.tight_layout()
plt.show()


## 5. optimal_value par point de données (combo vs baseline)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
colors = np.where(df["certified"], "#2a9d8f", "#e76f51")
ax.scatter(range(len(df)), df["optimal_value"], c=colors, alpha=0.5, s=10)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("Index de ligne (ordre du fichier)")
ax.set_ylabel("optimal_value")
ax.set_title("optimal_value par point de données (certifié = optimal_value > 0)")
plt.tight_layout()
plt.show()


## 6. Table brute triable/filtrable

In [ ]:
# Exemple : les points où le gain est le plus négatif (la stratégie fait moins bien que le baseline)
df.sort_values("gain").head(30)


## 7. Découverte de patterns (arbre de décision)

On entraîne un **arbre de décision** pour trouver des règles simples ("si largeur du
neurone 1 > X et l = 2 alors gain élevé") qui expliquent le `gain` et la
`certification`, à partir de features disponibles pour chaque point de données :
`l, u_idx, k, j_idx, LB/UB des 2 neurones, largeur des intervalles, data_index`.

**Attention à l'interprétation** : un arbre profond peut sur-apprendre (mémoriser le
bruit plutôt que capturer un vrai pattern). On limite volontairement la profondeur
(`max_depth`) pour ne garder que des règles robustes et lisibles — augmentez-la
prudemment si besoin, en gardant un œil sur le score de validation.


In [ ]:
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier, plot_tree, export_text
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, accuracy_score

# === Préparation des features ===
tree_df = df.dropna(subset=["gain", "LB_neuron1", "UB_neuron1", "LB_neuron2", "UB_neuron2"]).copy()
tree_df["width_neuron1"] = tree_df["UB_neuron1"] - tree_df["LB_neuron1"]
tree_df["width_neuron2"] = tree_df["UB_neuron2"] - tree_df["LB_neuron2"]
tree_df["width_product"] = tree_df["width_neuron1"] * tree_df["width_neuron2"]

FEATURE_COLS = [
    "l", "u_idx", "k", "j_idx",
    "LB_neuron1", "UB_neuron1", "LB_neuron2", "UB_neuron2",
    "width_neuron1", "width_neuron2", "width_product",
    "data_index",
]
FEATURE_COLS = [c for c in FEATURE_COLS if c in tree_df.columns]

X = tree_df[FEATURE_COLS].fillna(tree_df[FEATURE_COLS].median())
y_gain = tree_df["gain"]
y_cert = tree_df["certified"].astype(int)

print(f"{len(tree_df)} points de données utilisés, {len(FEATURE_COLS)} features : {FEATURE_COLS}")


### 7a. Arbre de régression — LB/UB seuls, puis avec `k` et `l`

Même logique que pour la classification : on regarde si `k` et `l` ajoutent du
pouvoir prédictif sur le `gain` au-delà des seules bornes des 2 neurones.
- **Modèle 1** : `LB_neuron1, UB_neuron1, LB_neuron2, UB_neuron2` uniquement.
- **Modèle 2** : Modèle 1 + `k, l`.


In [ ]:
MAX_DEPTH_REG = 3  # augmentez prudemment si besoin de règles plus fines

def fit_and_report_reg_tree(feature_cols, title):
    Xr = tree_df[feature_cols].fillna(tree_df[feature_cols].median())
    yr = tree_df["gain"]

    X_train, X_test, y_train, y_test = train_test_split(Xr, yr, test_size=0.25, random_state=0)

    reg = DecisionTreeRegressor(max_depth=MAX_DEPTH_REG, min_samples_leaf=10, random_state=0)
    reg.fit(X_train, y_train)

    r2_train = r2_score(y_train, reg.predict(X_train))
    r2_test = r2_score(y_test, reg.predict(X_test))

    print(f"--- {title} ---")
    print(f"Features: {feature_cols}")
    print(f"R² train: {r2_train:.3f} | R² test: {r2_test:.3f}")
    if r2_test < 0.1:
        print("[!] R² test faible : l'arbre n'explique presque rien du gain avec ces features.")

    fig, ax = plt.subplots(figsize=(20, 10))
    plot_tree(reg, feature_names=feature_cols, filled=True, rounded=True, fontsize=9, ax=ax, precision=2)
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

    print("\nRègles extraites :\n")
    print(export_text(reg, feature_names=feature_cols))
    print()

    return reg, r2_test


FEATURES_BOUNDS_ONLY = ["LB_neuron1", "UB_neuron1", "LB_neuron2", "UB_neuron2"]
FEATURES_BOUNDS_PLUS_KL = FEATURES_BOUNDS_ONLY + ["k", "l"]

reg_bounds, r2_bounds = fit_and_report_reg_tree(
    FEATURES_BOUNDS_ONLY, "Modèle 1 — LB/UB des 2 neurones uniquement"
)
reg_bounds_kl, r2_bounds_kl = fit_and_report_reg_tree(
    FEATURES_BOUNDS_PLUS_KL, "Modèle 2 — LB/UB + k, l"
)

print(f"Gain de R² en ajoutant k, l : {r2_bounds_kl - r2_bounds:+.3f}")


### 7b. Arbre de classification — LB/UB seuls, puis avec `k` et `l`

Deux modèles pour voir si la position du produit croisé (`l`, `k`) ajoute du pouvoir
prédictif au-delà des seules bornes des 2 neurones :
- **Modèle 1** : `LB_neuron1, UB_neuron1, LB_neuron2, UB_neuron2` uniquement.
- **Modèle 2** : Modèle 1 + `k, l`.


In [ ]:
MAX_DEPTH_CLF = 3

def fit_and_report_tree(feature_cols, title):
    Xc = tree_df[feature_cols].fillna(tree_df[feature_cols].median())
    yc = tree_df["certified"].astype(int)

    X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
        Xc, yc, test_size=0.25, random_state=0, stratify=yc
    )

    clf = DecisionTreeClassifier(max_depth=MAX_DEPTH_CLF, min_samples_leaf=10, random_state=0)
    clf.fit(X_train_c, y_train_c)

    acc_train = accuracy_score(y_train_c, clf.predict(X_train_c))
    acc_test = accuracy_score(y_test_c, clf.predict(X_test_c))
    baseline_acc = max(yc.mean(), 1 - yc.mean())

    print(f"--- {title} ---")
    print(f"Features: {feature_cols}")
    print(f"Accuracy train: {acc_train:.3f} | Accuracy test: {acc_test:.3f} | Baseline (classe majoritaire): {baseline_acc:.3f}")
    if acc_test <= baseline_acc + 0.02:
        print("[!] L'arbre ne fait pas mieux que la classe majoritaire avec ces features.")

    fig, ax = plt.subplots(figsize=(20, 10))
    plot_tree(
        clf, feature_names=feature_cols, class_names=["non certifié", "certifié"],
        filled=True, rounded=True, fontsize=9, ax=ax,
    )
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

    print("\nRègles extraites :\n")
    print(export_text(clf, feature_names=feature_cols))
    print()

    return clf, acc_test


FEATURES_BOUNDS_ONLY = ["LB_neuron1", "UB_neuron1", "LB_neuron2", "UB_neuron2"]
FEATURES_BOUNDS_PLUS_KL = FEATURES_BOUNDS_ONLY + ["k", "l"]

clf_bounds, acc_bounds = fit_and_report_tree(
    FEATURES_BOUNDS_ONLY, "Modèle 1 — LB/UB des 2 neurones uniquement"
)
clf_bounds_kl, acc_bounds_kl = fit_and_report_tree(
    FEATURES_BOUNDS_PLUS_KL, "Modèle 2 — LB/UB + k, l"
)

print(f"Gain d'accuracy en ajoutant k, l : {acc_bounds_kl - acc_bounds:+.3f}")


### 7c. Importance des features (les deux arbres)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

imp_reg_bounds = pd.Series(reg_bounds.feature_importances_, index=FEATURES_BOUNDS_ONLY).sort_values()
axes[0, 0].barh(imp_reg_bounds.index, imp_reg_bounds.values, color="#2a9d8f")
axes[0, 0].set_title("Régression (gain) — LB/UB seuls")

imp_reg_kl = pd.Series(reg_bounds_kl.feature_importances_, index=FEATURES_BOUNDS_PLUS_KL).sort_values()
axes[0, 1].barh(imp_reg_kl.index, imp_reg_kl.values, color="#2a9d8f")
axes[0, 1].set_title("Régression (gain) — LB/UB + k, l")

imp_bounds = pd.Series(clf_bounds.feature_importances_, index=FEATURES_BOUNDS_ONLY).sort_values()
axes[1, 0].barh(imp_bounds.index, imp_bounds.values, color="#264653")
axes[1, 0].set_title("Classification — LB/UB seuls")

imp_bounds_kl = pd.Series(clf_bounds_kl.feature_importances_, index=FEATURES_BOUNDS_PLUS_KL).sort_values()
axes[1, 1].barh(imp_bounds_kl.index, imp_bounds_kl.values, color="#e76f51")
axes[1, 1].set_title("Classification — LB/UB + k, l")

plt.tight_layout()
plt.show()


## 8. Frontières de décision de l'arbre, avec les points réels

Un arbre à plusieurs features ne se visualise pas directement en 2D. Pour voir
concrètement **où l'arbre trace ses coupures par rapport aux points**, on reprend,
pour chaque modèle, ses **2 features les plus importantes** et on ré-entraîne un
arbre 2D (même profondeur) juste pour l'affichage : le fond coloré montre la
prédiction de l'arbre sur la grille, les points sont les vraies données.

⚠️ C'est une **vue simplifiée** : l'arbre 2D ré-entraîné sur seulement 2 features
peut différer légèrement de l'arbre complet (3+ features) présenté en section 7 —
il sert à *illustrer* la logique de coupure, pas à remplacer l'arbre complet.


In [ ]:
def plot_2d_decision_regions(X_full, y, feature_names, is_classifier, max_depth, title):
    """Ré-entraîne un arbre sur exactement 2 features et affiche ses régions de
    décision (fond) superposées aux points réels."""
    f1, f2 = feature_names
    X2 = X_full[[f1, f2]]

    if is_classifier:
        model2d = DecisionTreeClassifier(max_depth=max_depth, min_samples_leaf=10, random_state=0)
    else:
        model2d = DecisionTreeRegressor(max_depth=max_depth, min_samples_leaf=10, random_state=0)
    model2d.fit(X2, y)

    pad1 = (X2[f1].max() - X2[f1].min()) * 0.05 or 1
    pad2 = (X2[f2].max() - X2[f2].min()) * 0.05 or 1
    xx, yy = np.meshgrid(
        np.linspace(X2[f1].min() - pad1, X2[f1].max() + pad1, 300),
        np.linspace(X2[f2].min() - pad2, X2[f2].max() + pad2, 300),
    )
    grid = pd.DataFrame({f1: xx.ravel(), f2: yy.ravel()})
    zz = model2d.predict(grid).reshape(xx.shape)

    fig, ax = plt.subplots(figsize=(8, 6))

    if is_classifier:
        ax.contourf(xx, yy, zz, levels=[-0.5, 0.5, 1.5], colors=["#fbe3dd", "#d7efe9"], alpha=0.8)
        colors = np.where(y == 1, "#2a9d8f", "#e76f51")
        ax.scatter(X2[f1], X2[f2], c=colors, s=12, alpha=0.7, edgecolor="none")
        handles = [
            plt.Line2D([0], [0], marker="o", color="w", markerfacecolor="#2a9d8f", label="certifié", markersize=7),
            plt.Line2D([0], [0], marker="o", color="w", markerfacecolor="#e76f51", label="non certifié", markersize=7),
        ]
        ax.legend(handles=handles)
    else:
        vmax = np.nanmax(np.abs(zz)) or 1
        cf = ax.contourf(xx, yy, zz, levels=20, cmap="RdYlGn", vmin=-vmax, vmax=vmax, alpha=0.75)
        fig.colorbar(cf, ax=ax, label="gain prédit (arbre 2D)")
        ax.scatter(X2[f1], X2[f2], c=y, cmap="RdYlGn", vmin=-vmax, vmax=vmax, s=14, edgecolor="black", linewidth=0.3)

    ax.set_xlabel(f1)
    ax.set_ylabel(f2)
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

    return model2d


### 8a. Régression (gain) — modèle LB/UB seuls

In [ ]:
top2_reg_bounds = imp_reg_bounds.sort_values(ascending=False).index[:2].tolist()
X_reg_bounds = tree_df[FEATURES_BOUNDS_ONLY].fillna(tree_df[FEATURES_BOUNDS_ONLY].median())
_ = plot_2d_decision_regions(
    X_reg_bounds, tree_df["gain"], top2_reg_bounds, is_classifier=False,
    max_depth=MAX_DEPTH_REG, title=f"Gain — arbre 2D sur {top2_reg_bounds}",
)


### 8b. Régression (gain) — modèle LB/UB + k, l

In [ ]:
top2_reg_kl = imp_reg_kl.sort_values(ascending=False).index[:2].tolist()
X_reg_kl = tree_df[FEATURES_BOUNDS_PLUS_KL].fillna(tree_df[FEATURES_BOUNDS_PLUS_KL].median())
_ = plot_2d_decision_regions(
    X_reg_kl, tree_df["gain"], top2_reg_kl, is_classifier=False,
    max_depth=MAX_DEPTH_REG, title=f"Gain — arbre 2D sur {top2_reg_kl}",
)


### 8c. Classification (certified) — modèle LB/UB seuls

In [ ]:
top2_clf_bounds = imp_bounds.sort_values(ascending=False).index[:2].tolist()
X_clf_bounds = tree_df[FEATURES_BOUNDS_ONLY].fillna(tree_df[FEATURES_BOUNDS_ONLY].median())
y_clf = tree_df["certified"].astype(int)
_ = plot_2d_decision_regions(
    X_clf_bounds, y_clf, top2_clf_bounds, is_classifier=True,
    max_depth=MAX_DEPTH_CLF, title=f"Certification — arbre 2D sur {top2_clf_bounds}",
)


### 8d. Classification (certified) — modèle LB/UB + k, l

In [ ]:
top2_clf_kl = imp_bounds_kl.sort_values(ascending=False).index[:2].tolist()
X_clf_kl = tree_df[FEATURES_BOUNDS_PLUS_KL].fillna(tree_df[FEATURES_BOUNDS_PLUS_KL].median())
_ = plot_2d_decision_regions(
    X_clf_kl, y_clf, top2_clf_kl, is_classifier=True,
    max_depth=MAX_DEPTH_CLF, title=f"Certification — arbre 2D sur {top2_clf_kl}",
)
